In [ ]:
# Load packages
import os

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
    "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

# COMMAND 

In [ ]:
%%writefile saige_gene_step01.sh
#!/bin/bash

# Exit immediately if a command exits with a non-zero status
set -o errexit

# Treat unset variables as an error (prevents using undefined variables)
set -o nounset

# Loop through the populations
for pop in EUR; do

    echo "Processing population: ${pop}"
    
    # Loop through the phenotypes 
    for pheno in aml_phenotype; do 

            step1_fitNULLGLMM.R \
                --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
                --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
                --plinkFile="${variance_files}/${pop}_array.forCate_vr" \
                --phenoFile="${phenotype}/${pheno}_files/EUR_deprivation_index_file.txt" \
                --useSparseGRMtoFitNULL=TRUE \
                --useSparseGRMforVarRatio=TRUE \
                --IsOverwriteVarianceRatioFile=TRUE \
                --isCateVarianceRatio=TRUE \
                --phenoCol=aml_phenotype \
                --traitType=binary \
                --covarColList="age,sex,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10" \
                --qCovarColList=sex \
                --sampleIDColinphenoFile=person_id \
                --outputPrefix="${output}/${pop}_${pheno}_restricted_deprivation_cohort_null_glmm" 

    done
done


# SUBMISSION 

In [ ]:
%%bash --out saige_gene_step01

# Source the dsub setup script
source ~/aou_dsub.bash 

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 20 \
  --preemptible \
  --machine-type n1-standard-1 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input-recursive phenotype="${WORKSPACE_BUCKET}/data/saige_phenotype_files/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive variance_files="${WORKSPACE_BUCKET}/data/saige_step01/variance_files/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --script "saige_gene_step01.sh"
